<a href="https://colab.research.google.com/github/parika8ec-hub/Assignment10/blob/main/Copy_of_Assignment10_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets torch -q

In [ ]:
!pip install transformers==5.5.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 62.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import transformers
print(transformers.__version__)

5.5.3


In [ ]:
from transformers import TrainingArguments
print(TrainingArguments)

<class 'transformers.training_args.TrainingArguments'>


In [ ]:
import inspect
print(inspect.signature(TrainingArguments.__init__))

(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_co

In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import TrainingArguments

from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer
)

# =========================
# 2. LOAD & PREPARE DATA
# =========================
#Load dataset
review_data = pd.read_csv('sample_data/amazon_alexa.tsv', sep='\t')

#display few rows of dataset
print('Few Rows of Amazon Alexa Customer Review Dataset:')
print(review_data.head())

#drop feedback and verifies_reviews columns and create new dataframe
df = review_data[['verified_reviews', 'feedback']].dropna()
#add new columns
df.columns = ['text', 'label']
df['label'] = df['label'].astype(int)#convert label column in integer

# Split dataset as 80% training and 20% test
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].astype(str),
    df['label'],
    test_size=0.2,
    random_state=42
)

# =========================
# 3. LOAD TOKENIZER
# =========================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# =========================
# 4. CONVERT TO DATASET
# =========================
train_dataset = Dataset.from_dict({
    "text": train_texts.tolist(),
    "label": train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    "text": test_texts.tolist(),
    "label": test_labels.tolist()
})

#Select small dataset to fast process
train_dataset = train_dataset.select(range(2000))
test_dataset = test_dataset.select(range(500))

# =========================
# 5. TOKENIZATION FUNCTION
# =========================
def tokenize_function(batch):
    return tokenizer(
        batch["text"],          # Takes the "text" field from the dataset batch
        padding="max_length",   # Pads all sequences to the same fixed length
        truncation=True,        # Truncates sequences longer than max_length
        max_length=128          # Sets maximum token length to 128 tokens
    )

# Apply tokenization to the training dataset
train_dataset = train_dataset.map(
    tokenize_function,   # Function that converts text into tokens (input IDs, attention masks)
    batched=True         # Processes multiple samples at once for faster execution
)

# Apply tokenization to the test dataset
test_dataset = test_dataset.map(
    tokenize_function,   # Same tokenization function applied to test data
    batched=True         # Enables batch processing for efficiency
)

# Remove text column (not needed anymore)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# Rename label column for Trainer
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

# Set format for PyTorch
train_dataset.set_format("torch")
test_dataset.set_format("torch")

# =========================
# 6. LOAD BERT MODEL
# =========================
# Load a pre-trained BERT model for text classification
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",  # Base BERT model trained on lowercase English text
    num_labels=2          # Number of output classes (e.g., positive / negative sentiment)
)

# =========================
# 7. METRICS FUNCTION
# =========================
def compute_metrics(eval_pred):
    # Unpack model outputs: logits (raw predictions) and true labels
    logits, labels = eval_pred

    # Convert logits into predicted class (highest probability index)
    preds = np.argmax(logits, axis=1)

    # Return evaluation metrics (here: accuracy only)
    return {
        "accuracy": accuracy_score(labels, preds)
    }

Few Rows of Amazon Alexa Customer Review Dataset:
   rating       date         variation  \
0       5  31-Jul-18  Charcoal Fabric    
1       5  31-Jul-18  Charcoal Fabric    
2       4  31-Jul-18    Walnut Finish    
3       5  31-Jul-18  Charcoal Fabric    
4       5  31-Jul-18  Charcoal Fabric    

                                    verified_reviews  feedback  
0                                      Love my Echo!         1  
1                                          Loved it!         1  
2  Sometimes while playing a game, you can answer...         1  
3  I have had a lot of fun with this thing. My 4 ...         1  
4                                              Music         1  


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# =========================
# 8. TRAINING ARGUMENTS
# =========================
training_args = TrainingArguments(
    output_dir="./results",              # Folder where model checkpoints and outputs will be saved

    eval_strategy="no",              # Not Run evaluation after each full epoch
    save_strategy="no",              # Not Save model after each epoch
    logging_strategy="epoch",           # Log training metrics after each epoch

    per_device_train_batch_size=8,     # Batch size for training (per GPU/CPU)
    per_device_eval_batch_size=8,      # Batch size for evaluation

    num_train_epochs=1,                 # Number of full passes through the dataset

    weight_decay=0.01,                   # Regularization to reduce overfitting
    fp16=True                    # faster training on GPU
)

In [ ]:
# =========================
# 9. TRAINER
# =========================
# Initialize the Trainer API for training and evaluation
trainer = Trainer(
    model=model,                       # The pre-trained BERT model for classification
    args=training_args,                # Training configuration (epochs, batch size, logging, etc.)
    train_dataset=train_dataset,       # Dataset used for training the model
    eval_dataset=test_dataset,         # Dataset used for evaluation during/after training
    compute_metrics=compute_metrics    # Function to calculate evaluation metrics (e.g., accuracy)
)

# =========================
# 10. TRAIN MODEL
# =========================
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
250,0.229715


TrainOutput(global_step=250, training_loss=0.22971531677246093, metrics={'train_runtime': 2489.0668, 'train_samples_per_second': 0.804, 'train_steps_per_second': 0.1, 'total_flos': 131555527680000.0, 'train_loss': 0.22971531677246093, 'epoch': 1.0})

In [ ]:
# =========================
# 11. EVALUATION
# =========================
y_true = test_dataset["labels"]
# Run predictions on the test dataset using the trained model
predictions = trainer.predict(test_dataset)

# Extract raw model outputs (logits) and convert them to class labels
y_pred = np.argmax(predictions.predictions, axis=1)

print(len(y_true), len(y_pred))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


500 500


In [ ]:
# Print header for readability
print("\nClassification Report:\n")

# Generate and print precision, recall, F1-score, and support
print(classification_report(y_true, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.78      0.60      0.67        47
           1       0.96      0.98      0.97       453

    accuracy                           0.95       500
   macro avg       0.87      0.79      0.82       500
weighted avg       0.94      0.95      0.94       500



**Interpretation:**

The model shows strong overall performance with an accuracy of 95%, meaning it correctly classifies most of the reviews. It performs extremely well on the majority class (class 1), with very high precision (0.96), recall (0.98), and F1-score (0.97), indicating it is very effective at identifying positive reviews. However, the performance on the minority class (class 0) is significantly weaker, with lower recall (0.60) and F1-score (0.67), meaning the model misses a considerable number of negative reviews. This imbalance is also reflected in the dataset distribution, where class 1 heavily dominates class 0. As a result, while the model appears highly accurate overall, it is biased toward the majority class and less reliable at detecting negative sentiment.